#  3  Carga al Data Warehouse
**Metodología HEFESTO — Paso 4: Integración de Datos**

Lee los parquet de `03_processed/` y los carga en PostgreSQL.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.load import cargar_desde_processed, ejecutar_carga, crear_engine, verificar_carga, cargar_config
import pandas as pd

##  Configuración


In [2]:
# Verificar que la conexión funciona antes de cargar
config = cargar_config("../config/settings.yaml")
engine = crear_engine(config)

[LOAD] ✅ Conexión exitosa a db.umbxwxsikjvqkybraipi.supabase.co:5432/postgres


##  Carga inicial

Usamos `modo="replace"` para la carga inicial (borra y recrea las tablas).
Para actualizaciones posteriores, cambiar a `modo="append"`.

In [ ]:
# CARGA INICIAL — reemplaza las tablas si ya existen
cargar_desde_processed(
    path_config="../config/settings.yaml",
    modo="replace",
)

## Verificación de integridad

Consultas de validación para asegurarnos que el DW quedó bien poblado.

In [ ]:
from sqlalchemy import text

# 1. Ventas por categoría
query = """
SELECT p.categoria,
       SUM(f.cantidad_vendida)  AS total_unidades,
       ROUND(SUM(f.monto_total)::numeric, 2) AS total_inr
FROM "factVentas" f
JOIN "dimProducto" p ON f."idProducto" = p."idProducto"
GROUP BY p.categoria
ORDER BY total_inr DESC
"""
with engine.connect() as con:
    df_test = pd.read_sql(query, con)
display(df_test)

In [ ]:
# 2. Ventas por estado geográfico (top 10)
query2 = """
SELECT g.estado,
       SUM(f.cantidad_vendida)  AS total_unidades,
       ROUND(SUM(f.monto_total)::numeric, 2) AS total_inr
FROM "factVentas" f
JOIN "dimGeografia" g ON f."idGeografia" = g."idGeografia"
GROUP BY g.estado
ORDER BY total_inr DESC
LIMIT 10
"""
with engine.connect() as con:
    df_test2 = pd.read_sql(query2, con)
display(df_test2)

In [ ]:
# 3. Comparativa Amazon vs Merchant
query3 = """
SELECT c.tipo_fulfillment,
       SUM(f.cantidad_vendida)   AS total_unidades,
       SUM(f.cantidad_cancelada) AS total_canceladas,
       ROUND(100.0 * SUM(f.cantidad_cancelada) /
             NULLIF(SUM(f.cantidad_vendida + f.cantidad_cancelada), 0), 2) AS tasa_cancelacion
FROM "factVentas" f
JOIN "dimCanal" c ON f."idCanal" = c."idCanal"
GROUP BY c.tipo_fulfillment
"""
with engine.connect() as con:
    df_test3 = pd.read_sql(query3, con)
display(df_test3)